# 🚀 Xyether Strong Series (v2.5 / v3) Headless Training Engine on Kaggle
### Autonomous Dual T4 GPU Training Pipeline

In [ ]:
# 1. Clone Project Repository & Setup Environment
import os, sys, subprocess

print("📡 Cloning Xyether Upscale Training Engine...")
!rm -rf /kaggle/working/engine
!git clone https://github.com/Thanawanit/upscale-training.git /kaggle/working/engine
%cd /kaggle/working/engine

# Install dependencies
!pip install -q pytorch-msssim kornia aria2
print("✅ Environment ready!")

In [ ]:
# 2. Prepare Transplanted Checkpoint (Zero-Init Head + Nearest Residual)
from weights.transplant import prepare_xyether_base

print("💉 Performing weight surgery on net_g_89000.pth...")
base_path = prepare_xyether_base(cache_dir="/kaggle/working/weights")
print(f"✅ Transplanted Base Ready: {base_path}")

In [ ]:
# 3. Run Phase 1: Fidelity & Denoise Base Training (8,000 iters)
import os

os.makedirs("/kaggle/working/output_models/phase1", exist_ok=True)

print("🔥 Starting Phase 1 Training...")
!python train_phase1.py \
    --data_dir /kaggle/working/dataset \
    --pretrained_path /kaggle/working/weights/xyether_transplanted_base.pth \
    --output_dir /kaggle/working/output_models/phase1 \
    --batch_size 16 \
    --patch_size 128 \
    --total_iters 8000 \
    --lr 1e-4 \
    --eval_every 1000 \
    --save_every 2000


In [ ]:
# 4. Run Phase 2: Sharp Polish Training (4,000 iters)
import glob

p1_ckpts = sorted(glob.glob("/kaggle/working/output_models/phase1/xyether_phase1_step_*.pth"))
latest_p1 = p1_ckpts[-1] if p1_ckpts else "/kaggle/working/weights/xyether_transplanted_base.pth"
print(f"Loading Phase 1 Checkpoint: {latest_p1}")

os.makedirs("/kaggle/working/output_models/phase2", exist_ok=True)

print("✨ Starting Phase 2 Sharp Polish (FFLoss + D-UNet)...")
!python train_phase2.py \
    --data_dir /kaggle/working/dataset \
    --pretrained_g "{latest_p1}" \
    --output_dir /kaggle/working/output_models/phase2 \
    --batch_size 12 \
    --patch_size 128 \
    --total_iters 4000 \
    --lr_g 3e-5 \
    --lr_d 3e-5 \
    --ffl_weight 0.05 \
    --gan_weight 0.10 \
    --eval_every 1000 \
    --save_every 2000


In [ ]:
# 5. Copy Production Models to Output Root for 1-Click Download
import shutil, os

final_pth = "/kaggle/working/output_models/phase2/Xyether_Strong_v3_Final.pth"
if os.path.exists(final_pth):
    shutil.copy(final_pth, "/kaggle/working/Xyether_Strong_v3_Final.pth")
    print("🎉 Final Model ready in /kaggle/working/Xyether_Strong_v3_Final.pth!")